In [ ]:
__author__="Kushvinth"

# Basic Module Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import random
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import albumentations as A



# SAFE TIFF LOADER (Works in Kaggle)



In [ ]:

def load_3d_tiff(path):
    img = Image.open(path)
    slices = []
    try:
        for i in range(10000):
            img.seek(i)
            slices.append(np.array(img))
    except EOFError:
        pass
    return np.stack(slices, axis=0)  # (Z,Y,X)


# Dataset

In [ ]:

class VesuviusDataset(Dataset):
    def __init__(self, img_paths, mask_paths=None, patch_size=(64,128,128), transforms=None, mode='train'):
        self.img_paths = img_paths
        self.mask_paths = mask_paths
        self.patch_size = patch_size
        self.transforms = transforms
        self.mode = mode

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img = load_3d_tiff(self.img_paths[idx]).astype(np.float32)

        if self.mask_paths is not None:
            mask = load_3d_tiff(self.mask_paths[idx]).astype(np.uint8)
        else:
            mask = np.zeros_like(img, dtype=np.uint8)

        Z,Y,X = img.shape
        pz,py,px = self.patch_size

        z0 = random.randint(0, max(0,Z-pz))
        y0 = random.randint(0, max(0,Y-py))
        x0 = random.randint(0, max(0,X-px))

        img = img[z0:z0+pz, y0:y0+py, x0:x0+px]
        mask = mask[z0:z0+pz, y0:y0+py, x0:x0+px]

        if self.transforms:
            xs, ys = [], []
            for i in range(img.shape[0]):
                aug = self.transforms(image=img[i].astype(np.uint8), mask=mask[i].astype(np.uint8))
                xs.append(aug['image'])
                ys.append(aug['mask'])
            img = np.stack(xs)
            mask = np.stack(ys)

        img = img.astype(np.float32) / 255.0
        mask = (mask == 1).astype(np.float32)

        return (
            torch.tensor(img[None], dtype=torch.float32),
            torch.tensor(mask[None], dtype=torch.float32)
        )



# UNET 3D MODEL (Compact version)

In [ ]:
class Conv3dBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.net(x)

class UpConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_ch, out_ch, 2, 2)
    def forward(self, x): return self.up(x)

class UNet3D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, features=[16,32,64,128]):
        super().__init__()
        self.encs, self.pools = nn.ModuleList(), nn.ModuleList()

        for f in features:
            self.encs.append(Conv3dBlock(in_ch, f))
            in_ch = f
            self.pools.append(nn.MaxPool3d(2))

        self.bottleneck = Conv3dBlock(features[-1], features[-1]*2)

        self.upconvs, self.decs = nn.ModuleList(), nn.ModuleList()
        for f in reversed(features):
            self.upconvs.append(UpConv(features[-1]*2 if f==features[-1] else prev_f, f))
            self.decs.append(Conv3dBlock(f*2, f))
            prev_f = f

        self.final = nn.Conv3d(features[0], out_ch, kernel_size=1)

    def forward(self, x):
        skips = []
        for enc, pool in zip(self.encs, self.pools):
            x = enc(x)
            skips.append(x)
            x = pool(x)

        x = self.bottleneck(x)

        for up, dec, skip in zip(self.upconvs, self.decs, reversed(skips)):
            x = up(x)
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = torch.cat([x, skip], dim=1)
            x = dec(x)

        return self.final(x)

# Loss

In [ ]:
class DiceLoss(nn.Module):
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        num = 2 * (probs * targets).sum() + 1e-6
        den = probs.sum() + targets.sum() + 1e-6
        return 1 - num / den

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25):
        super().__init__()
        self.gamma, self.alpha = gamma, alpha
    def forward(self, logits, targets):
        p = torch.sigmoid(logits)
        pt = p*targets + (1-p)*(1-targets)
        w = self.alpha*targets + (1-self.alpha)*(1-targets)
        return (-w*(1-pt)**self.gamma * torch.log(pt+1e-8)).mean()

class CombinedLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.dice = DiceLoss()
        self.focal = FocalLoss()
    def forward(self, logits, targets):
        return self.dice(logits, targets) + 0.75*self.focal(logits, targets)



# Sliding Window Inference

In [ ]:
def sliding_window(volume, model, patch=(64,128,128), stride=(32,64,64)):
    model.eval()
    Z,Y,X = volume.shape
    pz,py,px = patch
    sz,sy,sx = stride

    out = np.zeros((Z,Y,X), np.float32)
    cnt = np.zeros_like(out)

    with torch.no_grad():
        for z in range(0, Z-pz+1, sz):
            for y in range(0, Y-py+1, sy):
                for x in range(0, X-px+1, sx):
                    patch_data = volume[z:z+pz, y:y+py, x:x+px]
                    patch_data = torch.tensor(patch_data[None,None]/255.0, dtype=torch.float32).cuda()
                    pred = torch.sigmoid(model(patch_data))[0,0].cpu().numpy()
                    out[z:z+pz, y:y+py, x:x+px] += pred
                    cnt[z:z+pz, y:y+py, x:x+px] += 1

    return out / np.maximum(cnt,1)



# Setup: Paths / Splits

In [ ]:
ROOT = Path("/kaggle/input/vesuvius-challenge-surface-detection")

train_imgs = sorted((ROOT/"train_images").glob("*.tif"))
train_masks = sorted((ROOT/"train_labels").glob("*.tif"))

split = int(0.9 * len(train_imgs))
train_list = train_imgs[:split]
val_list = train_imgs[split:]
train_mask_list = train_masks[:split]
val_mask_list = train_masks[split:]


# DataLoaders

In [ ]:
transforms = A.Compose([
    A.RandomBrightnessContrast(p=0.5),
    A.HorizontalFlip(p=0.5),
])

train_ds = VesuviusDataset(train_list, train_mask_list, transforms=transforms)
val_ds = VesuviusDataset(val_list, val_mask_list, transforms=None)

train_dl = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=2)
val_dl = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2)


In [ ]:
from torch.amp import autocast        # NEW API
from torch.cuda.amp import GradScaler # Still valid

model = UNet3D().cuda()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

# Correct initialization
scaler = GradScaler()
criterion = CombinedLoss()

EPOCHS = 3

def validate(model, loader):
    model.eval()
    dices = []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.cuda(), y.cuda()
            pred = torch.sigmoid(model(x))
            pred = (pred > 0.5).float()
            dice = (2*(pred*y).sum() + 1e-6) / (pred.sum()+y.sum()+1e-6)
            dices.append(dice.item())
    return np.mean(dices)

best_dice = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for x, y in tqdm(train_dl):
        x, y = x.cuda(), y.cuda()
        optimizer.zero_grad()

        # NEW CORRECT AUTOTCAST
        with autocast("cuda"):
            out = model(x)
            loss = criterion(out, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    val_dice = validate(model, val_dl)
    print(f"Epoch {epoch} | TrainLoss={total_loss/len(train_dl):.4f} | ValDice={val_dice:.4f}")

    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(model.state_dict(), "/kaggle/working/best_model.pth")
        print("Saved best model!")


# Inference on Test Set

In [ ]:
def rle_encode(mask):
    pixels = mask.flatten(order="F")
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 2
    runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)

test_df = pd.read_csv(ROOT/"test.csv")
test_images = ROOT/"test_images"

model.load_state_dict(torch.load("/kaggle/working/best_model.pth"))
model.eval()

import os
import zipfile
import tifffile as tiff

os.makedirs("/kaggle/working/pred_masks", exist_ok=True)
test_df = pd.read_csv(ROOT/"test.csv")
test_imgs = ROOT/"test_images"

model.load_state_dict(torch.load("/kaggle/working/best_model.pth"))
model.eval()

for vid in tqdm(test_df.id.values):

    # Load test volume (safe loader)
    vol = load_3d_tiff(test_imgs/f"{vid}.tif").astype(np.float32)
    vol_norm = (vol - vol.mean()) / (vol.std() + 1e-8)

    # Sliding window prediction
    prob = sliding_window(vol_norm, model)

    # Binarize → uint8 mask (0/1)
    mask = (prob > 0.5).astype("uint8")

    # Save mask with SAME SHAPE + TYPE
    out_path = f"/kaggle/working/pred_masks/{vid}.tif"
    tiff.imwrite(out_path, mask, dtype="uint8")

print("All TIFF masks written successfully.")
zip_path = "/kaggle/working/submission.zip"

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for vid in test_df.id.values:
        file_path = f"/kaggle/working/pred_masks/{vid}.tif"
        z.write(file_path, arcname=f"{vid}.tif")

print("Submission created:", zip_path)
